# ClustMRF on TREC Robust04 — Nested 5-Fold CV with SVM^rank

**Paper:** *Ranking Document Clusters using Markov Random Fields* (Raiber & Kurland, SIGIR 2013)

**Corpus:** TREC Disk 4 + Disk 5 (FBIS, FT, FR94, LA Times) — ~500 K documents

**Topics / Qrels:** TREC Robust04 — 249 queries (301–450, 601–700)

**Metrics (primary):** MAP, P@5, P@10, NDCG@5, NDCG@10

---

## Experimental protocol

| Stage | Detail |
|-------|--------|
| Initial retrieval | SDM (λ_T=0.85, λ_O=0.10, λ_U=0.05) + DirichletLM (μ=2500) — fixed, not tuned |
| ClustMRF features | 13 non-web features: 4 lQD/lQC query-similarity + 9 lC doc-quality |
| Weight learning | SVM^rank (Joachims 2006), linear kernel, C tuned per outer fold |
| CV structure | **Outer:** 5-fold on 249 queries → no test-query information leaks into training |
| | **Inner:** 5-fold on the 4 training folds → tunes SVM^rank C ∈ {0.001, 0.01, 0.1, 1.0, 10.0} |
| Cluster relevance | NDCG@k of cluster members ordered by sim(Q,d) — continuous label per paper §3/fn.7 |
| Cluster unrolling | §4.1: best→worst cluster, within cluster by sim(Q,d) |

No test query touches the training pipeline at any point.

> **Paper comparison note:** The paper reports MAP@50 (cutoff = |D_init| = 50). This notebook uses
> n_docs=100 and evaluates MAP@1000 (full list). MAP@50 is added in §6 for direct comparison.

## 1  Environment Setup

In [1]:
import os, sys, re, subprocess, tempfile, pathlib, warnings, logging
warnings.filterwarnings('ignore')

# Use the symlink path (not canonical) so PyTerrier's version-check regex doesn't
# falsely match the '25' in '11.0.25' as JDK 25.
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11'

ROOT = pathlib.Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import KFold

import ir_measures
from ir_measures import MAP, nDCG, P

import pyterrier as pt
if not pt.java.started():
    pt.java.set_memory_limit(8192)   # 8 GB
    pt.java.init()

from src.algorithms.clust_mrf import ClustMRF, FEAT_NAMES

logging.basicConfig(level=logging.WARNING)
print(f'PyTerrier  : {pt.__version__}')
print(f'ir_measures: {ir_measures.__version__}')
print(f'Features   : {len(FEAT_NAMES)} ({FEAT_NAMES})')

PyTerrier  : 1.0.4
ir_measures: 0.4.3
Features   : 13 (['geo_qsim', 'stdv_qsim', 'max_qsim', 'min_qsim', 'min_dsim', 'max_dsim', 'geo_dsim', 'min_icompress', 'geo_icompress', 'max_icompress', 'geo_entropy', 'min_entropy', 'max_entropy'])


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## 2  Paths

In [2]:
# ── Corpus ─────────────────────────────────────────────────────────────────
DISK4 = pathlib.Path('/mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC/Disk4')
DISK5 = pathlib.Path('/mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC/Disk5')

# ── Queries / Qrels ────────────────────────────────────────────────────────
TOPICS_PATH = pathlib.Path('/mnt/bi-strg4/pool1-data/DATASETS/i/ds3400/lv_ibm_strg/IBM_STORAGE/USERS_DATA/annabel/qrels_queries/queriesROBUST.xml')
QRELS_PATH  = pathlib.Path('/mnt/bi-strg4/pool1-data/DATASETS/i/ds3400/lv_ibm_strg/IBM_STORAGE/USERS_DATA/gadm/qrels.robust2004')

# ── SVM^rank binaries ──────────────────────────────────────────────────────
_SVMR_BASE = pathlib.Path('/mnt/bi-strg4/pool1-data/DATASETS/i/ds3400/lv_ibm_strg/IBM_STORAGE/USERS_DATA/liorab/ieir21/supporting_evidence_code')
SVM_LEARN    = _SVMR_BASE / 'svm_rank_learn'
SVM_CLASSIFY = _SVMR_BASE / 'svm_rank_classify'

# ── Local artefacts ────────────────────────────────────────────────────────
INDEX_DIR = ROOT / 'data' / 'indexes' / 'robust04'
RUNS_DIR  = ROOT / 'data' / 'runs'    / 'robust04'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

for label, p in [
    ('Disk4',         DISK4),
    ('Disk5',         DISK5),
    ('Topics',        TOPICS_PATH),
    ('Qrels',         QRELS_PATH),
    ('svm_rank_learn',    SVM_LEARN),
    ('svm_rank_classify', SVM_CLASSIFY),
]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'{status}  {label}: {p}')

✓  Disk4: /mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC/Disk4
✓  Disk5: /mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC/Disk5
✓  Topics: /mnt/bi-strg4/pool1-data/DATASETS/i/ds3400/lv_ibm_strg/IBM_STORAGE/USERS_DATA/annabel/qrels_queries/queriesROBUST.xml
✓  Qrels: /mnt/bi-strg4/pool1-data/DATASETS/i/ds3400/lv_ibm_strg/IBM_STORAGE/USERS_DATA/gadm/qrels.robust2004
✓  svm_rank_learn: /mnt/bi-strg4/pool1-data/DATASETS/i/ds3400/lv_ibm_strg/IBM_STORAGE/USERS_DATA/liorab/ieir21/supporting_evidence_code/svm_rank_learn
✓  svm_rank_classify: /mnt/bi-strg4/pool1-data/DATASETS/i/ds3400/lv_ibm_strg/IBM_STORAGE/USERS_DATA/liorab/ieir21/supporting_evidence_code/svm_rank_classify


## 3  Build Robust04 Index (Disk 4 + Disk 5)

In [ ]:
# ── SGML parser ────────────────────────────────────────────────────────────
_DOC_RE  = re.compile(r'<DOC>(.*?)</DOC>', re.DOTALL)
_DOCNO_RE = re.compile(r'<DOCNO>\s*(.*?)\s*</DOCNO>')
_TEXT_RE  = re.compile(
    r'<(?:TEXT|HEADLINE|TI|DOCTITLE|TITLE|H[1-6])>(.*?)</(?:TEXT|HEADLINE|TI|DOCTITLE|TITLE|H[1-6])>',
    re.DOTALL | re.IGNORECASE,
)
_TAG_RE   = re.compile(r'<[^>]+>')

def _parse_sgml(raw: str):
    """Yield {docno, text} dicts from a TREC SGML string."""
    for m in _DOC_RE.finditer(raw):
        body = m.group(1)
        dn   = _DOCNO_RE.search(body)
        if not dn:
            continue
        parts = _TEXT_RE.findall(body)
        text  = ' '.join(p.strip() for p in parts)
        text  = _TAG_RE.sub(' ', text)
        text  = re.sub(r'\s+', ' ', text).strip()
        if text:
            yield {'docno': dn.group(1).strip(), 'text': text}


ROBUST04_SUBDIRS = ['FBIS', 'FT', 'FR94', 'LATIMES']


def _robust04_files():
    """Yield all corpus files from Disk4 and Disk5 (Robust04 subcollections).

    Handles three file types found in the wild:
      - *.Z         — standard gzip (FBIS, FT, LATIMES)
      - *.0Z *.1Z *.2Z — gzip with numeric prefix (FR94)
      - no extension — plain SGML (e.g. Disk4/FT/FT911_1)
    """
    for disk in [DISK4, DISK5]:
        for sub in ROBUST04_SUBDIRS:
            subpath = disk / sub
            if not subpath.exists():
                continue
            for f in sorted(subpath.rglob('*')):
                if not f.is_file():
                    continue
                name = f.name.upper()
                if 'READ' in name or name.startswith('.'):
                    continue
                # files ending in Z cover .Z, .0Z, .1Z, .2Z;
                # files with no dot cover plain SGML like FT911_1
                if name.endswith('Z') or '.' not in f.name:
                    yield f


def corpus_iter():
    """Stream unique Robust04 documents from corpus files."""
    seen         = set()
    corpus_files = list(_robust04_files())
    for f in tqdm(corpus_files, desc='Streaming corpus files', unit='file'):
        proc = subprocess.Popen(['zcat', str(f)],
                                stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        raw = proc.stdout.read().decode('latin-1', errors='replace')
        if proc.wait() != 0 or not raw.strip():
            # Plain (uncompressed) SGML — e.g. FT911_1
            raw = f.read_bytes().decode('latin-1', errors='replace')
        for doc in _parse_sgml(raw):
            if doc['docno'] not in seen:
                seen.add(doc['docno'])
                yield doc

total_files = sum(1 for _ in _robust04_files())
print(f'Total corpus files to index: {total_files}')


In [ ]:
INDEX_DIR.mkdir(parents=True, exist_ok=True)
props_file    = INDEX_DIR / 'data.properties'
blocks_marker = INDEX_DIR / '.blocks_enabled'

def _props_has_inverted(pfile):
    return pfile.exists() and 'index.inverted.class' in pfile.read_text()

def _fix_inverted_index():
    import struct
    from jnius import autoclass
    IndexOnDisk = autoclass('org.terrier.structures.IndexOnDisk')
    CF          = autoclass('org.terrier.structures.indexing.CompressionFactory')
    BIB         = autoclass('org.terrier.structures.indexing.classical.BlockInvertedIndexBuilder')
    FSOGen      = autoclass('org.terrier.structures.FSOMapFileLexiconGeneric')
    TextFact    = autoclass('org.terrier.structures.seralization.FixedSizeTextFactory')
    BLEFact     = autoclass('org.terrier.structures.BasicLexiconEntry$Factory')
    idx = IndexOnDisk.createIndex(str(INDEX_DIR), 'data')
    cfg = CF.getCompressionConfiguration('inverted', [], True, False)
    print('  Running BlockInvertedIndexBuilder...')
    BIB(idx, 'inverted', cfg).createInvertedIndex()
    idx.close()
    print('  Inverted index built.')
    # Fix properties written by the builder (num.Tokens reset to 0, charmap shortcut)
    txt = props_file.read_text()
    txt = txt.replace('num.Tokens=0\n', 'num.Tokens=92708312\n')
    txt = txt.replace('index.lexicon.bsearchshortcut=charmap\n',
                      'index.lexicon.bsearchshortcut=default\n')
    props_file.write_text(txt)
    # Rebuild fsomapid: termId -> byte_offset in fsomapfile
    lex_file = INDEX_DIR / 'data.lexicon.fsomapfile'
    id_file  = INDEX_DIR / 'data.lexicon.fsomapid'
    file_size = lex_file.stat().st_size
    mapfile = FSOGen.loadMapFile(str(lex_file), TextFact('20'), BLEFact(), 'fileinmem')
    n_terms  = mapfile.size()
    rec_size = file_size // n_terms
    offsets  = [0] * n_terms
    it = mapfile.entrySet().iterator()
    i  = 0
    while it.hasNext():
        tid = it.next().getValue().getTermId()
        if 0 <= tid < n_terms:
            offsets[tid] = i * rec_size
        i += 1
    with open(id_file, 'wb') as fh:
        for off in offsets:
            fh.write(struct.pack('>q', off))
    print(f'  Rebuilt {id_file.name}: {id_file.stat().st_size:,} bytes')

if props_file.exists() and not blocks_marker.exists():
    import shutil
    print('Existing index lacks position blocks. Rebuilding...')
    shutil.rmtree(str(INDEX_DIR))
    INDEX_DIR.mkdir(parents=True, exist_ok=True)

if not props_file.exists():
    print('Building Robust04 index (with blocks) -- ~10-25 min on first run...')
    indexer = pt.IterDictIndexer(
        str(INDEX_DIR),
        overwrite  = True,
        meta       = {'docno': 30, 'text': 131072},  # 128 KB — accommodates full Robust04 docs
        text_attrs = ['text'],
        blocks     = True,
        tokeniser  = 'EnglishTokeniser',
        stemmer    = 'PorterStemmer',
        stopwords  = 'terrier',
    )
    indexer.index(corpus_iter())
    blocks_marker.touch()
    if not _props_has_inverted(props_file):
        print('Inverted index not built by IterDictIndexer -- running repair...')
        _fix_inverted_index()
    print('Index built.')
elif not _props_has_inverted(props_file):
    print('Inverted index missing -- repairing...')
    _fix_inverted_index()
else:
    print(f'Index (with blocks) already exists at {INDEX_DIR} -- loading.')

index = pt.IndexFactory.of(str(props_file))
stats = index.getCollectionStatistics()
print(f'Documents    : {stats.numberOfDocuments:,}')
print(f'Tokens       : {stats.numberOfTokens:,}')
print(f'Unique terms : {stats.numberOfUniqueTerms:,')

## 4  Topics & Qrels

In [5]:
# ── Topics: strip Indri #combine(...) wrapper ──────────────────────────────
import xml.etree.ElementTree as ET

tree = ET.parse(str(TOPICS_PATH))
root = tree.getroot()

def _clean_query(text: str) -> str:
    m = re.match(r'#combine\((.*)\)', text.strip())
    return m.group(1).strip() if m else text.strip()

topics_list = []
for q in root.findall('query'):
    num  = q.findtext('number', '').strip()
    text = _clean_query(q.findtext('text', '').strip())
    if num and text:
        topics_list.append({'qid': num, 'query': text})

topics_df = pd.DataFrame(topics_list).sort_values('qid').reset_index(drop=True)

# ── Qrels (standard TREC format: qid 0 docid rel) ─────────────────────────
qrels_rows = []
with open(QRELS_PATH) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 4:
            qid, _, docid, rel = parts
            qrels_rows.append({'query_id': qid, 'doc_id': docid, 'relevance': int(rel)})

qrels_df   = pd.DataFrame(qrels_rows)
judged_qids = set(qrels_df['query_id'].unique())

# Keep only queries with qrels
topics_df = topics_df[topics_df['qid'].isin(judged_qids)].reset_index(drop=True)

# Per-query docno → relevance dict (for cluster relevance labelling)
qid_docrel: dict[str, dict[str, int]] = (
    qrels_df[qrels_df['relevance'] > 0]
    .groupby('query_id')
    .apply(lambda g: dict(zip(g['doc_id'], g['relevance'])))
    .to_dict()
)

print(f'Queries : {len(topics_df)}')
print(f'Qrel rows: {len(qrels_df):,}')
print(f'Rel distribution:\n{qrels_df["relevance"].value_counts().sort_index().to_string()}')
topics_df.head()

Queries : 249
Qrel rows: 311,410
Rel distribution:
relevance
0    293998
1     16381
2      1031


,qid,query
0,301,international organized crime
1,302,poliomyelitis post polio
2,303,hubble telescope achievements
3,304,endangered species mammals
4,305,dangerous vehicles


## 5  Initial Retrieval (SDM, fixed weights)

SDM weights follow Metzler & Croft 2005 and the paper's setup: λ_T=0.85, λ_O=0.10, λ_U=0.05.  
DirichletLM with μ=2500.

We also run BM25 as a weaker baseline (no tuning — default b=0.75, k1=1.2).

In [6]:
import time

def _save_run(run_df: pd.DataFrame, path: pathlib.Path, tag: str) -> None:
    with open(path, 'w') as f:
        for qid, grp in run_df.sort_values(['qid', 'rank']).groupby('qid'):
            for rank, row in enumerate(grp.itertuples(), 1):
                f.write(f'{qid} Q0 {row.docno} {rank} {row.score:.6f} {tag}\n')
    print(f'Saved -> {path.relative_to(ROOT)}')

def _load_run(path: pathlib.Path) -> pd.DataFrame:
    rows = []
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) >= 5:
                rows.append({'qid': p[0], 'docno': p[2], 'rank': int(p[3]), 'score': float(p[4])})
    return pd.DataFrame(rows)


# ---- BM25 ----------------------------------------------------------------
_bm25_path   = RUNS_DIR / 'bm25.txt'
_bm25_parquet = RUNS_DIR / 'bm25_with_text.parquet'
if _bm25_parquet.exists():
    print('Loading cached BM25 run (parquet)...')
    bm25_run = pd.read_parquet(_bm25_parquet)
elif _bm25_path.exists():
    print('BM25 TREC file found but no parquet. Re-running for texts...')
    bm25_br  = pt.BatchRetrieve(index, wmodel='BM25', num_results=1000,
                                 metadata=['docno', 'text'],
                                 controls={'bm25.b': 0.75, 'bm25.k_1': 1.2})
    bm25_run = bm25_br.transform(topics_df)
    bm25_run.to_parquet(_bm25_parquet)
else:
    bm25_br  = pt.BatchRetrieve(index, wmodel='BM25', num_results=1000,
                                 metadata=['docno', 'text'],
                                 controls={'bm25.b': 0.75, 'bm25.k_1': 1.2})
    t0 = time.time()
    bm25_run = bm25_br.transform(topics_df)
    print(f'BM25 done in {time.time()-t0:.1f}s')
    _save_run(bm25_run, _bm25_path, 'BM25')
    bm25_run.to_parquet(_bm25_parquet)


# ---- SDM (initial retrieval for ClustMRF) --------------------------------
_sdm_path    = RUNS_DIR / 'sdm.txt'
_sdm_parquet = RUNS_DIR / 'sdm_with_text.parquet'

def _build_sdm_run():
    sdm_rewrite  = pt.rewrite.SDM()
    dirichlet_br = pt.BatchRetrieve(
        index, wmodel='DirichletLM', num_results=1000,
        metadata=['docno', 'text'],
        controls={'c': 2500},
    )
    t0  = time.time()
    run = (sdm_rewrite >> dirichlet_br).transform(topics_df)
    print(f'SDM done in {time.time()-t0:.1f}s')
    return run

if _sdm_parquet.exists():
    print('Loading cached SDM run (parquet)...')
    sdm_run = pd.read_parquet(_sdm_parquet)
elif _sdm_path.exists():
    print('SDM TREC file found but no parquet. Re-running for texts...')
    sdm_run = _build_sdm_run()
    _save_run(sdm_run, _sdm_path, 'SDM_DirichletLM')
    sdm_run.to_parquet(_sdm_parquet)
else:
    sdm_run = _build_sdm_run()
    _save_run(sdm_run, _sdm_path, 'SDM_DirichletLM')
    sdm_run.to_parquet(_sdm_parquet)

print(f'\nBM25 run : {len(bm25_run):,} rows, {bm25_run["qid"].nunique()} queries')
print(f'SDM run  : {len(sdm_run):,} rows, {sdm_run["qid"].nunique()} queries')
sdm_run.head()

Loading cached BM25 run (parquet)...


Loading cached SDM run (parquet)...



BM25 run : 240,115 rows, 249 queries
SDM run  : 240,115 rows, 249 queries


,qid,docid,docno,text,rank,score,query,query_0
0,301,311620,FBIS4-40260,Text of Presidential Edict on Organized Crime ...,0,9.281637,international organized crime #combine:0=0.1:w...,international organized crime
1,301,318094,FBIS4-46734,North Caucasus Anticrime Chief Views Current T...,1,9.258489,international organized crime #combine:0=0.1:w...,international organized crime
2,301,233768,FBIS3-23986,Seminar on Criminology Held 1990-1993 Crime Fi...,2,9.232747,international organized crime #combine:0=0.1:w...,international organized crime
3,301,340161,FBIS4-68801,Khabarovsk Authorities Fight Organized Crime C...,3,8.061973,international organized crime #combine:0=0.1:w...,international organized crime
4,301,313199,FBIS4-41839,"Militia General on Organized Crime, Anticrime ...",4,7.721235,international organized crime #combine:0=0.1:w...,international organized crime


## 6  Baseline Evaluation (BM25 & SDM)

In [ ]:
MEASURES = [MAP, P @ 5, P @ 10, nDCG @ 5, nDCG @ 10, nDCG @ 20]

bm25_eval = bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
sdm_eval  = sdm_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})

bm25_agg = ir_measures.calc_aggregate(MEASURES, qrels_df, bm25_eval)
sdm_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, sdm_eval)

print('=== Baselines — Robust04 (249 queries) ===')
print(f'{"Measure":<18} {"BM25":>10} {"SDM":>10}')
print('=' * 40)
for m in MEASURES:
    print(f'  {str(m):<16} {float(bm25_agg[m]):>10.4f} {float(sdm_agg[m]):>10.4f}')

# MAP@50 for direct comparison with the paper (paper computes MAP at cutoff 50 = |D_init|)
bm25_agg50 = ir_measures.calc_aggregate([MAP @ 50], qrels_df, bm25_eval)
sdm_agg50  = ir_measures.calc_aggregate([MAP @ 50], qrels_df, sdm_eval)
print(f'\n  {"AP@50 (paper cmp)":<16} {float(bm25_agg50[MAP@50]):>10.4f} {float(sdm_agg50[MAP@50]):>10.4f}')
print(f'  {"AP@50 paper (init)":16} {"n/a":>10} {"≈0.199":>10}')

## 7  ClustMRF Feature Extraction

For every query we pre-compute the 13-feature cluster matrix **once** and cache it.  
The cross-validation loop then reads from this cache — no re-computation per fold.

In [ ]:
# k=5, n_docs=100: use top-100 documents as the initial candidate set.
N_DOCS = 100
K_CLUSTER = 5

extractor = ClustMRF(index=index, k=K_CLUSTER, n_docs=N_DOCS)

# qid_data[qid] = {'feats': ndarray(n,13), 'cluster_nn': list[ndarray],
#                  'sim_qd': ndarray(n), 'top_df': DataFrame}
qid_data: dict[str, dict] = {}

for qid, grp in tqdm(sdm_run.groupby('qid'), desc='Extracting features', unit='query'):
    feats, cluster_nn, sim_qd, top_df = extractor.extract_cluster_features(
        grp.reset_index(drop=True)
    )
    qid_data[qid] = {
        'feats':      feats,
        'cluster_nn': cluster_nn,
        'sim_qd':     sim_qd,
        'top_df':     top_df,
    }

# Summarise
n_feat_total = sum(d['feats'].shape[0] for d in qid_data.values())
print(f'Queries with features : {len(qid_data)}')
print(f'Total cluster vectors : {n_feat_total:,}')
print(f'Feature shape example : {next(iter(qid_data.values()))["feats"].shape}')

## 8  Nested 5-Fold Cross-Validation with SVM^rank

**Protocol (no data leakage):**

```
for outer_fold in 5:
    test_qids  ← ~50 queries  (never touched during training)
    train_qids ← ~199 queries
    for inner_fold in 5:          # hyper-parameter tuning
        val_qids        ← ~40 queries
        inner_train_qids ← ~159 queries
        for C in {0.001, 0.01, 0.1, 1.0, 10.0}:
            train SVM^rank on inner_train_qids
            score val_qids → MAP
    best_C ← argmax mean MAP over inner folds
    train SVM^rank on train_qids with best_C
    score test_qids → test results
```

In [ ]:
import math

def _cluster_ndcg(nn_docnos, sim_qd_cluster, doc_rels):
    """NDCG@k for a cluster, ordering members by descending sim(Q,d).

    This is the cluster score used in SVM^rank training per the paper
    (§3 / footnote 7): 'the NDCG@k of the k constituent documents of a
    cluster serves as the cluster score used for ranking clusters in the
    learning phase.'  Binary relevance: rel(d) = 1 if qrel > 0, else 0.
    """
    order = np.argsort(sim_qd_cluster)[::-1]
    rels = [1 if doc_rels.get(nn_docnos[j], 0) > 0 else 0 for j in order]
    n_rel = sum(rels)
    if n_rel == 0:
        return 0.0
    dcg  = sum(r / math.log2(i + 2) for i, r in enumerate(rels))
    idcg = sum(1.0 / math.log2(i + 2) for i in range(n_rel))
    return dcg / idcg


def _write_svmrank_file(fpath, qids, qid_data, qid_docrel, qid_int_map):
    """
    Write an SVM^rank input file for the given query list.
    Returns offsets: {qid: (start_line_idx, n_clusters)}.
    Cluster relevance = NDCG@k of member docs ordered by sim(Q,d).
    """
    offsets = {}
    idx     = 0
    with open(fpath, 'w') as f:
        for qid in qids:
            data       = qid_data[qid]
            feats      = data['feats']
            cluster_nn = data['cluster_nn']
            sim_qd     = data['sim_qd']
            top_df     = data['top_df']
            n          = len(feats)
            offsets[qid] = (idx, n)
            q_int      = qid_int_map[qid]
            doc_rels   = qid_docrel.get(qid, {})
            for i in range(n):
                nn             = cluster_nn[i]
                nn_docnos      = [top_df.iloc[j]['docno'] for j in nn]
                sim_qd_cluster = sim_qd[nn]
                c_rel          = _cluster_ndcg(nn_docnos, sim_qd_cluster, doc_rels)
                feat_str       = ' '.join(f'{j+1}:{v:.8f}' for j, v in enumerate(feats[i]))
                f.write(f'{c_rel} qid:{q_int} {feat_str}\n')
            idx += n
    return offsets


def _run_svmrank(train_file, test_file, C, tmpdir):
    """
    Train on train_file, classify test_file.
    Returns list of float scores (one per line of test_file).
    """
    model_file = tmpdir / f'model_C{C}.dat'
    pred_file  = tmpdir / f'pred_C{C}.dat'
    train_res  = subprocess.run(
        [str(SVM_LEARN), '-c', str(C), '-t', '0',
         str(train_file), str(model_file)],
        capture_output=True, text=True,
    )
    if train_res.returncode != 0:
        raise RuntimeError(f'svm_rank_learn failed (C={C}): {train_res.stderr[-300:]}')

    classify_res = subprocess.run(
        [str(SVM_CLASSIFY), str(test_file), str(model_file), str(pred_file)],
        capture_output=True, text=True,
    )
    if classify_res.returncode != 0:
        raise RuntimeError(f'svm_rank_classify failed: {classify_res.stderr[-300:]}')

    with open(pred_file) as fh:
        return [float(line.strip()) for line in fh if line.strip()]


def _parse_predictions(raw_preds, qids, offsets):
    """Split flat prediction list back into {qid: ndarray}."""
    result = {}
    for qid in qids:
        start, n = offsets[qid]
        result[qid] = np.array(raw_preds[start:start + n])
    return result


def _scores_to_run(qids, qid_data, qid_scores, full_sdm=None):
    """
    Unroll cluster scores → ranked document list.
    §4.1: clusters sorted best→worst; within cluster docs by sim(Q,d).
    If full_sdm is given, docs ranked beyond n_docs are appended in their
    original SDM order (with scores below 0) so MAP is computed over the
    full 1000-doc list, not just the reranked top-N window.
    """
    tail_by_qid = {}
    if full_sdm is not None:
        sdm_grpd = {
            qid: grp.sort_values('rank').reset_index(drop=True)
            for qid, grp in full_sdm.groupby('qid')
        }
        for qid in qids:
            if qid in sdm_grpd:
                n_top = qid_data[qid]['feats'].shape[0]
                tail  = sdm_grpd[qid].iloc[n_top:]
                if not tail.empty:
                    tail_by_qid[qid] = tail

    rows = []
    for qid in qids:
        data       = qid_data[qid]
        scores     = np.array(qid_scores[qid])
        cluster_nn = data['cluster_nn']
        sim_qd     = data['sim_qd']
        top_df     = data['top_df']
        n          = len(scores)

        sorted_centers = np.argsort(scores)[::-1]
        seen, result   = set(), []
        for ci in sorted_centers:
            nn = cluster_nn[ci]
            for j in nn[np.argsort(sim_qd[nn])[::-1]]:
                if j not in seen:
                    result.append(int(j))
                    seen.add(j)
        for j in range(n):
            if j not in seen:
                result.append(j)

        for rank, idx in enumerate(result):
            rows.append({
                'qid':   qid,
                'docno': top_df.iloc[idx]['docno'],
                'rank':  rank + 1,
                'score': float(n - rank),
            })

        tail = tail_by_qid.get(qid)
        if tail is not None:
            for i, tail_row in enumerate(tail.itertuples()):
                rows.append({
                    'qid':   qid,
                    'docno': tail_row.docno,
                    'rank':  n + i + 1,
                    'score': -float(i),
                })

    return pd.DataFrame(rows)


def _eval_map(run_df, qrels_df):
    ev  = run_df.rename(columns={'qid': 'query_id', 'docno': 'doc_id'})
    agg = ir_measures.calc_aggregate([MAP], qrels_df, ev)
    return float(agg[MAP])


print('SVM^rank helpers defined.')

In [ ]:
OUTER_K  = 5
INNER_K  = 5
C_VALUES = [0.001, 0.01, 0.1, 1.0, 10.0]

qids     = sorted(qid_data.keys())
qid_int_map = {qid: i + 1 for i, qid in enumerate(qids)}   # str → int (SVM^rank qid must be int)

outer_kf   = KFold(n_splits=OUTER_K, shuffle=True, random_state=42)
qids_arr   = np.array(qids)

cv_fold_runs = []
best_C_per_fold = []

with tempfile.TemporaryDirectory(prefix='clustmrf_robust04_') as tmpdir_str:
    tmpdir = pathlib.Path(tmpdir_str)

    for outer_fold, (tr_idx, te_idx) in enumerate(outer_kf.split(qids_arr)):
        train_qids = qids_arr[tr_idx].tolist()
        test_qids  = qids_arr[te_idx].tolist()
        print(f'\n=== Outer fold {outer_fold+1}/{OUTER_K}  '
              f'(train={len(train_qids)}, test={len(test_qids)}) ===')

        # ── Inner CV to tune C ──────────────────────────────────────────
        inner_kf    = KFold(n_splits=INNER_K, shuffle=True, random_state=0)
        train_arr   = np.array(train_qids)
        c_map_scores = {C: [] for C in C_VALUES}

        for inner_fold, (itr_idx, val_idx) in enumerate(inner_kf.split(train_arr)):
            inner_train_q = train_arr[itr_idx].tolist()
            val_q         = train_arr[val_idx].tolist()

            itr_file = tmpdir / f'o{outer_fold}_i{inner_fold}_train.dat'
            val_file = tmpdir / f'o{outer_fold}_i{inner_fold}_val.dat'

            _write_svmrank_file(itr_file, inner_train_q, qid_data, qid_docrel, qid_int_map)
            val_offsets = _write_svmrank_file(val_file, val_q, qid_data, qid_docrel, qid_int_map)

            # Filter qrels to validation queries only so MAP is not diluted
            val_qrels = qrels_df[qrels_df['query_id'].isin(val_q)]

            for C in C_VALUES:
                try:
                    preds    = _run_svmrank(itr_file, val_file, C, tmpdir)
                    val_sc   = _parse_predictions(preds, val_q, val_offsets)
                    val_run  = _scores_to_run(val_q, qid_data, val_sc, full_sdm=sdm_run)
                    val_map  = _eval_map(val_run, val_qrels)
                    c_map_scores[C].append(val_map)
                except Exception as exc:
                    print(f'  inner {inner_fold+1} C={C} failed: {exc}')
                    c_map_scores[C].append(0.0)

        # Pick best C by mean inner-fold MAP
        best_C = max(C_VALUES, key=lambda C: np.mean(c_map_scores[C]))
        best_C_per_fold.append(best_C)
        print(f'  Inner-fold MAP by C:')
        for C in C_VALUES:
            marker = ' ←' if C == best_C else ''
            print(f'    C={C:<8} MAP={np.mean(c_map_scores[C]):.4f}{marker}')

        # ── Train on all train_qids with best_C, score test_qids ────────
        full_train_file = tmpdir / f'o{outer_fold}_fulltrain.dat'
        test_file       = tmpdir / f'o{outer_fold}_test.dat'

        _write_svmrank_file(full_train_file, train_qids, qid_data, qid_docrel, qid_int_map)
        test_offsets = _write_svmrank_file(test_file, test_qids, qid_data, qid_docrel, qid_int_map)

        preds    = _run_svmrank(full_train_file, test_file, best_C, tmpdir)
        test_sc  = _parse_predictions(preds, test_qids, test_offsets)
        fold_run = _scores_to_run(test_qids, qid_data, test_sc, full_sdm=sdm_run)

        # Filter qrels to test queries for accurate fold-level MAP logging
        test_qrels = qrels_df[qrels_df['query_id'].isin(test_qids)]
        fold_map = _eval_map(fold_run, test_qrels)
        print(f'  Test MAP (fold {outer_fold+1}) = {fold_map:.4f}')
        cv_fold_runs.append(fold_run)

# Combine all held-out predictions
clustmrf_cv_run = pd.concat(cv_fold_runs, ignore_index=True)
print(f'\nCV run: {len(clustmrf_cv_run):,} rows, {clustmrf_cv_run["qid"].nunique()} queries')
print(f'Best C per outer fold: {best_C_per_fold}')

## 9  ClustMRF Baseline (Fixed Paper Weights, No CV)

For comparison: ClustMRF with the Table-3 heuristic weights from the paper — no training, no CV.  
Applied to all 249 queries at once (no fold structure).

In [11]:
_cm_fixed_path = RUNS_DIR / 'clustmrf_fixed.txt'
if _cm_fixed_path.exists():
    print('Loading cached ClustMRF (fixed weights) run…')
    clustmrf_fixed_run = _load_run(_cm_fixed_path)
else:
    cm_fixed = ClustMRF(index=index, k=K_CLUSTER, n_docs=N_DOCS)  # default weights
    t0 = time.time()
    clustmrf_fixed_run = cm_fixed.transform(sdm_run)
    print(f'Done in {time.time()-t0:.1f}s')
    _save_run(clustmrf_fixed_run, _cm_fixed_path, 'ClustMRF_fixed')

print(f'ClustMRF (fixed): {len(clustmrf_fixed_run):,} rows')

Loading cached ClustMRF (fixed weights) run…


ClustMRF (fixed): 240,115 rows


## 10  Results Comparison

In [ ]:
clustmrf_cv_eval    = clustmrf_cv_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
clustmrf_fixed_eval = clustmrf_fixed_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})

cv_agg    = ir_measures.calc_aggregate(MEASURES, qrels_df, clustmrf_cv_eval)
fixed_agg = ir_measures.calc_aggregate(MEASURES, qrels_df, clustmrf_fixed_eval)

rows = []
for name, agg in [
    ('BM25',                      bm25_agg),
    ('SDM (initial, fixed λ)',    sdm_agg),
    ('ClustMRF (fixed weights)',  fixed_agg),
    ('ClustMRF (5×5-fold CV)',    cv_agg),
]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    rows.append(row)

results_df = pd.DataFrame(rows).set_index('System')

results_df.loc['Δ (SDM − BM25)']                  = (
    results_df.loc['SDM (initial, fixed λ)'] - results_df.loc['BM25'])
results_df.loc['Δ (ClustMRF fixed − SDM)']        = (
    results_df.loc['ClustMRF (fixed weights)'] - results_df.loc['SDM (initial, fixed λ)'])
results_df.loc['Δ (ClustMRF CV − fixed weights)'] = (
    results_df.loc['ClustMRF (5×5-fold CV)'] - results_df.loc['ClustMRF (fixed weights)'])
results_df.loc['Δ (ClustMRF CV − SDM)']           = (
    results_df.loc['ClustMRF (5×5-fold CV)'] - results_df.loc['SDM (initial, fixed λ)'])

print('=== Robust04 (249 queries) — Full Results ===')
print(results_df.to_string())

# MAP@50: direct comparison with paper Table 2 (ROBUST row, Init=19.9, ClustMRF=21.0)
MEASURES50 = [MAP @ 50]
rows50 = []
for name, ev in [
    ('BM25',                     bm25_eval),
    ('SDM (initial, fixed λ)',   sdm_eval),
    ('ClustMRF (fixed weights)', clustmrf_fixed_eval),
    ('ClustMRF (5×5-fold CV)',   clustmrf_cv_eval),
]:
    agg50 = ir_measures.calc_aggregate(MEASURES50, qrels_df, ev)
    rows50.append({'System': name, 'AP@50': round(float(agg50[MAP @ 50]), 4)})

print('\n=== MAP@50 — direct paper comparison (paper ROBUST row: Init=0.199, ClustMRF=0.210) ===')
print(pd.DataFrame(rows50).set_index('System').to_string())

## 10.1  Interpolation: ClustMRF + BM25 / SDM

```
score(d, q) = α · score_ClustMRF + (1 − α) · score_base
```
Scores are min-max normalised per query before blending. Only docs present in both runs are retained.

Both **fixed-weight** and **CV-learned** ClustMRF variants are blended against BM25 and SDM.

In [ ]:
def minmax_norm(run_df: pd.DataFrame) -> pd.DataFrame:
    run_df = run_df.copy()
    grp    = run_df.groupby('qid')['score']
    lo     = grp.transform('min')
    hi     = grp.transform('max')
    run_df['score'] = (run_df['score'] - lo) / (hi - lo).replace(0, 1.0)
    return run_df


def interpolate_runs(run_a: pd.DataFrame, run_b: pd.DataFrame,
                     alpha: float) -> pd.DataFrame:
    a = run_a[['qid', 'docno', 'score']].rename(columns={'score': 'score_a'})
    b = run_b[['qid', 'docno', 'score']].rename(columns={'score': 'score_b'})
    merged = pd.merge(a, b, on=['qid', 'docno'])
    merged['score'] = alpha * merged['score_a'] + (1.0 - alpha) * merged['score_b']
    merged = (merged[['qid', 'docno', 'score']]
              .sort_values(['qid', 'score'], ascending=[True, False]))
    merged['rank'] = merged.groupby('qid').cumcount() + 1
    return merged.reset_index(drop=True)


bm25_norm        = minmax_norm(bm25_run)
sdm_norm         = minmax_norm(sdm_run)
clustmrf_norm    = minmax_norm(clustmrf_fixed_run)
clustmrf_cv_norm = minmax_norm(clustmrf_cv_run)

ALPHAS = [0.1, 0.25, 0.5, 0.75, 0.9]

interp_rows = []
for name, agg in [
    ('BM25',                   bm25_agg),
    ('SDM',                    sdm_agg),
    ('ClustMRF (fixed)',       fixed_agg),
    ('ClustMRF (CV)',          cv_agg),
]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    interp_rows.append(row)
interp_rows.append({'System': '---', **{str(m): '---' for m in MEASURES}})

for cm_tag, cm_norm in [('fixed', clustmrf_norm), ('CV', clustmrf_cv_norm)]:
    for alpha in ALPHAS:
        for base_tag, base_norm in [('BM25', bm25_norm), ('SDM', sdm_norm)]:
            run_i = interpolate_runs(cm_norm, base_norm, alpha)
            ev_i  = run_i.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
            agg_i = ir_measures.calc_aggregate(MEASURES, qrels_df, ev_i)
            row   = {'System': f'ClustMRF({cm_tag})+{base_tag}  α={alpha}'}
            for m in MEASURES:
                row[str(m)] = round(float(agg_i[m]), 4)
            interp_rows.append(row)
    interp_rows.append({'System': '---', **{str(m): '---' for m in MEASURES}})

interp_df = pd.DataFrame(interp_rows).set_index('System')
print('=== Robust04: Interpolation (α = weight on ClustMRF) ===')
print(interp_df.to_string())

## 11  Per-Query MAP Analysis

In [13]:
sdm_perq = {r.query_id: r.value
            for r in ir_measures.iter_calc([MAP], qrels_df, sdm_eval)
            if r.measure == MAP}
cv_perq  = {r.query_id: r.value
            for r in ir_measures.iter_calc([MAP], qrels_df, clustmrf_cv_eval)
            if r.measure == MAP}

perq_df = pd.DataFrame([
    {'qid': qid,
     'SDM_MAP':      round(sdm_perq.get(qid, 0.0), 4),
     'ClustMRF_MAP': round(cv_perq.get(qid, 0.0),  4)}
    for qid in sorted(sdm_perq)
])
perq_df['Δ MAP'] = (perq_df['ClustMRF_MAP'] - perq_df['SDM_MAP']).round(4)

wins   = (perq_df['Δ MAP'] > 0).sum()
losses = (perq_df['Δ MAP'] < 0).sum()
ties   = (perq_df['Δ MAP'] == 0).sum()

print(f'ClustMRF-CV vs SDM per-query MAP:')
print(f'  Wins   : {wins}')
print(f'  Losses : {losses}')
print(f'  Ties   : {ties}')
print()
print('Top-5 gains:')
print(perq_df.nlargest(5, 'Δ MAP')[['qid','SDM_MAP','ClustMRF_MAP','Δ MAP']].to_string(index=False))
print()
print('Top-5 losses:')
print(perq_df.nsmallest(5, 'Δ MAP')[['qid','SDM_MAP','ClustMRF_MAP','Δ MAP']].to_string(index=False))

ClustMRF-CV vs SDM per-query MAP:
  Wins   : 125
  Losses : 114
  Ties   : 10

Top-5 gains:
qid  SDM_MAP  ClustMRF_MAP  Δ MAP
678   0.1054        0.2699 0.1645
369   0.3260        0.4864 0.1604
603   0.2537        0.3950 0.1413
630   0.6294        0.7544 0.1250
615   0.1238        0.2318 0.1080

Top-5 losses:
qid  SDM_MAP  ClustMRF_MAP   Δ MAP
328   0.6011        0.3416 -0.2595
664   0.4071        0.1939 -0.2132
380   0.4060        0.2389 -0.1671
692   0.4160        0.3055 -0.1105
606   0.2987        0.2074 -0.0913


## 12  Save Runs (TREC Format)

In [14]:
_save_run(bm25_run,           RUNS_DIR / 'bm25.txt',            'BM25_Terrier')
_save_run(sdm_run,            RUNS_DIR / 'sdm.txt',             'SDM_DirichletLM')
_save_run(clustmrf_fixed_run, RUNS_DIR / 'clustmrf_fixed.txt',  'ClustMRF_fixed')
_save_run(clustmrf_cv_run,    RUNS_DIR / 'clustmrf_cv.txt',     'ClustMRF_CV_SVMrank')

Saved -> data/runs/robust04/bm25.txt


Saved -> data/runs/robust04/sdm.txt


Saved -> data/runs/robust04/clustmrf_fixed.txt


Saved -> data/runs/robust04/clustmrf_cv.txt


---

## Summary

| System | MAP | P@5 | P@10 | NDCG@10 |
|--------|-----|-----|------|--------|
| BM25 | *see §6* | | | |
| SDM (fixed λ) | *see §6* | | | |
| ClustMRF (fixed weights) | *see §10* | | | |
| ClustMRF (5×5-fold CV, SVM^rank) | *see §10* | | | |

**Key design choices:**
- **SDM weights are fixed** (0.85 / 0.10 / 0.05) — same as the paper's baseline.
- **ClustMRF feature weights are learned** per outer fold with SVM^rank, C tuned via inner 5-fold.
- **No leakage**: test queries are held out during both inner and outer training stages.
- **Cluster relevance** = `max(qrel[d] > 0 for d in cluster)` — binary label for Robust04.